<!-- NOTEBOOK_OVERVIEW -->
# 1. Semantic Baseline Evaluation (BGE + Logistic Regression)

## 2. Introduction
This notebook trains and evaluates the semantic-only baseline used as the main reference point for the hybrid-routing experiments. It tests how far a lightweight sentence-embedding classifier can go on its own before lexical or structural augmentation is introduced.

## 3. Workflow Steps
1. Load the split-specific processed dataset and assemble train/val/test/OOD prompt sets.
2. Encode prompts with `BAAI/bge-small-en-v1.5` sentence embeddings.
3. Train a class-balanced logistic regression classifier on the ID training split.
4. Select a validation threshold `t*` on `VAL` using the shared evaluation policy.
5. Evaluate the fixed model on `VAL`, `TEST`, and all configured OOD sets.
6. Export canonical semantic metrics and optional difficulty-bin diagnostics.

## 4. Evaluation and Protocol Notes
1. This notebook currently contributes the `deployment_threshold` semantic baseline rows used in repeatability and hybrid comparison tables.
2. Reported metrics include:
   - `accuracy`, `macro_f1`
   - `ROC-AUC`, `AUC-PR`
   - `TPR@1% FPR`, `TPR@5% FPR`, `TPR@10% FPR`
   - difficulty-bin summaries over prompt length and lexical complexity
3. The decision threshold is chosen on `VAL` only and then frozen for `TEST` and OOD evaluation.
4. No OOD tuning or calibration is performed in this notebook; OOD is evaluation-only.

## 5. Execution Notes
1. Run once per split tag (`A`, `B`, `C`) to populate the semantic baseline rows used elsewhere.
2. Keep threshold/evaluation policy aligned with notebooks 04 and 06 so comparisons remain valid.


In [1]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# 1) Imports + config

from pathlib import Path
import sys
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [2]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# 2) Load processed dataset v2

import os

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.evaluation.eval_metrics import (
    evaluate_predictions,
    results_to_dataframe,
    best_threshold_by_macro_f1,
)
from src.common.notebook_utils import encode_texts, safe_qcut, text_stats

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

# ============================
# DATASET SPLIT SELECTION
# Split A = original v2 (current baseline)
# Split B/C = repeatability resamples (same rows, new ID train/val/test, OOD fixed)
# ============================

SPLIT_TAG = os.getenv("SPLIT_TAG", "B").strip().upper()   # env override: A/B/C
WRITE_MINIMAL_OUTPUTS = os.getenv("WRITE_MINIMAL_OUTPUTS", "1").strip().lower() not in {"0", "false", "no"}
print("WRITE_MINIMAL_OUTPUTS:", WRITE_MINIMAL_OUTPUTS)

expected_filename_by_split = {
    "A": "jailbreak_benchmarks_processed_v2.csv",
    "B": "jailbreak_benchmarks_processed_v2_splitB.csv",
    "C": "jailbreak_benchmarks_processed_v2_splitC.csv",
}
if SPLIT_TAG not in expected_filename_by_split:
    raise ValueError("SPLIT_TAG must be 'A', 'B', or 'C'.")
processed_filename = expected_filename_by_split[SPLIT_TAG]

processed_path = DATA_PROCESSED / processed_filename
print("Using dataset:", processed_path)

df = pd.read_csv(processed_path)

print("Rows:", len(df))
print("\nSplit counts:")
print(df["split"].value_counts())
print("\nLabel counts:")
print(df["label"].value_counts())


WRITE_MINIMAL_OUTPUTS: False
Using dataset: /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/data/processed/jailbreak_benchmarks_processed_v2_splitC.csv
Rows: 6424

Split counts:
split
ood_test_injection_standard    3986
ood_test                        768
train                           694
ood_test_injection              678
test                            149
val                             149
Name: count, dtype: int64

Label counts:
label
1    3437
0    2987
Name: count, dtype: int64


In [3]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# Split views

df_train = df[df["split"] == "train"].copy()
df_val   = df[df["split"] == "val"].copy()
df_test  = df[df["split"] == "test"].copy()

OOD_SPLIT_ORDER = ["ood_test", "ood_test_injection", "ood_test_injection_standard"]
df_ood_map = {}
for split_name in OOD_SPLIT_ORDER:
    d = df[df["split"] == split_name].copy()
    if not d.empty:
        df_ood_map[split_name] = d

if "ood_test" not in df_ood_map:
    raise ValueError("Missing required split 'ood_test'.")

df_ood = df_ood_map["ood_test"]
df_ood_injection = df_ood_map.get("ood_test_injection")

for name, d in [("train", df_train), ("val", df_val), ("test", df_test)] + list(df_ood_map.items()):
    print(f"{name:18s}", d.shape, d["label"].value_counts().to_dict())

X_train_text = df_train["prompt_text"].astype(str).tolist()
X_val_text   = df_val["prompt_text"].astype(str).tolist()
X_test_text  = df_test["prompt_text"].astype(str).tolist()
X_ood_text   = df_ood["prompt_text"].astype(str).tolist()

y_train = df_train["label"].values
y_val   = df_val["label"].values
y_test  = df_test["label"].values
y_ood   = df_ood["label"].values
y_ood_map = {k: v["label"].values for k, v in df_ood_map.items()}


train              (694, 16) {1: 504, 0: 190}
val                (149, 16) {1: 109, 0: 40}
test               (149, 16) {1: 108, 0: 41}
ood_test           (768, 16) {0: 384, 1: 384}
ood_test_injection (678, 16) {0: 339, 1: 339}
ood_test_injection_standard (3986, 16) {1: 1993, 0: 1993}


In [4]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# 3) Embeddings: Compute

sem_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

X_sem_train = encode_texts(sem_model, X_train_text)
X_sem_val   = encode_texts(sem_model, X_val_text)
X_sem_test  = encode_texts(sem_model, X_test_text)

X_sem_ood_map = {
    split_name: encode_texts(sem_model, df_ood_map[split_name]["prompt_text"].astype(str).tolist())
    for split_name in df_ood_map
}
X_sem_ood = X_sem_ood_map["ood_test"]

print()
print("Embedding shapes:")
print("Train:", X_sem_train.shape)
print("Val:  ", X_sem_val.shape)
print("Test: ", X_sem_test.shape)
for split_name, X in X_sem_ood_map.items():
    print(f"{split_name:18s}", X.shape)



Embedding shapes:
Train: (694, 384)
Val:   (149, 384)
Test:  (149, 384)
ood_test           (768, 384)
ood_test_injection (678, 384)
ood_test_injection_standard (3986, 384)


In [5]:
# Cell Purpose: Build or load semantic feature representations for baseline/hybrid models.
# 4) Train semantic-only classifier

sem_clf = LogisticRegression(
    max_iter=5000,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

sem_clf.fit(X_sem_train, y_train)


LogisticRegression(class_weight='balanced', max_iter=5000, n_jobs=-1,
                   random_state=42)

In [6]:
# Cell Purpose: Evaluate model behavior and collect security-relevant performance metrics.
# Choose threshold t* on VAL (macro-F1) ONLY

val_proba = sem_clf.predict_proba(X_sem_val)[:, 1]
t_star, best_val_f1 = best_threshold_by_macro_f1(y_val, val_proba, n_grid=1001)

val_pred_at_t = (val_proba >= t_star).astype(int)
val_neg = (y_val == 0)
val_fpr_at_t = float(((val_pred_at_t == 1) & val_neg).sum() / max(val_neg.sum(), 1))

print(f"\nSelected threshold on VAL (macro-F1): t* = {t_star:.3f}")
print(f"Best VAL macro-F1 at t*: {best_val_f1:.4f}")
print(f"VAL FPR at t*: {val_fpr_at_t:.4f}")



Selected threshold on VAL (macro-F1): t* = 0.407
Best VAL macro-F1 at t*: 0.8601
VAL FPR at t*: 0.1500


In [7]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# 5) Evaluate VAL/TEST/OOD using frozen eval protocol

def eval_split(split_name, X, y_true, note_suffix=""):
    y_score = sem_clf.predict_proba(X)[:, 1]
    y_pred = (y_score >= t_star).astype(int)

    note = f"policy=macro_f1; t*={t_star:.3f}; val_fpr={val_fpr_at_t:.4f}; score=semantic_proba"
    if note_suffix:
        note = f"{note}; {note_suffix}"

    return evaluate_predictions(
        split_name=split_name,
        y_true=y_true,
        y_pred=y_pred,
        y_score_for_metrics=y_score,
        print_report=True,
        threshold_note=note,
    )

val_res  = eval_split("VAL",  X_sem_val,  y_val)
test_res = eval_split("TEST", X_sem_test, y_test)
ood_res  = eval_split("OOD",  X_sem_ood_map["ood_test"], y_ood_map["ood_test"], note_suffix="ood_name=ood_test")

results_df = results_to_dataframe("SEM_LOGREG_BGE", [val_res, test_res, ood_res])
results_df["eval_track"] = "deployment_threshold"
results_df["ood_name"] = "id"
results_df.loc[results_df["split"] == "OOD", "ood_name"] = "ood_test"

semantic_extra_ood_results = {}
for ood_name, X_ood_curr in X_sem_ood_map.items():
    if ood_name == "ood_test":
        continue
    y_ood_curr = y_ood_map[ood_name]
    ood_extra_res = eval_split("OOD", X_ood_curr, y_ood_curr, note_suffix=f"ood_name={ood_name}")
    df_extra = results_to_dataframe("SEM_LOGREG_BGE", [ood_extra_res])
    df_extra["eval_track"] = "deployment_threshold"
    df_extra["ood_name"] = ood_name
    semantic_extra_ood_results[ood_name] = df_extra

print()
print("=== Semantic Baseline Summary (primary OOD) ===")
display(results_df)
if semantic_extra_ood_results:
    print()
    print("=== Semantic Baseline Secondary OOD Rows ===")
    display(pd.concat(list(semantic_extra_ood_results.values()), ignore_index=True))



=== VAL ===
              precision    recall  f1-score   support

           0      0.756     0.850     0.800        40
           1      0.942     0.899     0.920       109

    accuracy                          0.886       149
   macro avg      0.849     0.875     0.860       149
weighted avg      0.892     0.886     0.888       149

Confusion matrix [ [TN FP] [FN TP] ]:
 [[34  6]
 [11 98]]
AUC-PR:  0.9822
ROC-AUC: 0.9491
TPR @ FPR: 1%=0.7248, 5%=0.8257, 10%=0.8624

=== TEST ===
              precision    recall  f1-score   support

           0      0.800     0.878     0.837        41
           1      0.952     0.917     0.934       108

    accuracy                          0.906       149
   macro avg      0.876     0.897     0.886       149
weighted avg      0.910     0.906     0.907       149

Confusion matrix [ [TN FP] [FN TP] ]:
 [[36  5]
 [ 9 99]]
AUC-PR:  0.9817
ROC-AUC: 0.9562
TPR @ FPR: 1%=0.5278, 5%=0.7130, 10%=0.9074

=== OOD ===
              precision    recall  f1-

,model,split,acc,macro_f1,auc_pr,roc_auc,tpr_at_1pct_fpr,tpr_at_5pct_fpr,tpr_at_10pct_fpr,semantic_coverage,defer_rate,threshold_note,eval_track,ood_name
0,SEM_LOGREG_BGE,VAL,0.885906,0.860094,0.982212,0.949083,0.724771,0.825688,0.862385,None,None,policy=macro_f1; t*=0.407; val_fpr=0.1500; sco...,deployment_threshold,id
1,SEM_LOGREG_BGE,TEST,0.906040,0.885586,0.981656,0.956188,0.527778,0.712963,0.907407,None,None,policy=macro_f1; t*=0.407; val_fpr=0.1500; sco...,deployment_threshold,id
2,SEM_LOGREG_BGE,OOD,0.718750,0.717023,0.840778,0.833584,0.203125,0.440104,0.562500,None,None,policy=macro_f1; t*=0.407; val_fpr=0.1500; sco...,deployment_threshold,ood_test



=== Semantic Baseline Secondary OOD Rows ===


,model,split,acc,macro_f1,auc_pr,roc_auc,tpr_at_1pct_fpr,tpr_at_5pct_fpr,tpr_at_10pct_fpr,semantic_coverage,defer_rate,threshold_note,eval_track,ood_name
0,SEM_LOGREG_BGE,OOD,0.646018,0.637149,0.648602,0.675107,0.014749,0.135693,0.212389,None,None,policy=macro_f1; t*=0.407; val_fpr=0.1500; sco...,deployment_threshold,ood_test_injection
1,SEM_LOGREG_BGE,OOD,0.729303,0.723672,0.767357,0.802753,0.068741,0.238334,0.411440,None,None,policy=macro_f1; t*=0.407; val_fpr=0.1500; sco...,deployment_threshold,ood_test_injection_standard


In [8]:
# Cell Purpose: Configure and validate split-specific data partitions and runtime settings.
# 6) Save metrics outputs

OUT_DIR = PROJECT_ROOT / "experiments" / "results" / "metrics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Keep canonical output unchanged for primary OOD.
out_path = OUT_DIR / f"metrics_semantic_split{SPLIT_TAG}.csv"
results_df.to_csv(out_path, index=False)
print()
print(f"Saved primary OOD metrics: {out_path}")

# Write secondary OOD rows as suffixed files only in full-output mode.
if not WRITE_MINIMAL_OUTPUTS:
    for ood_name, df_extra in semantic_extra_ood_results.items():
        out_extra = OUT_DIR / f"metrics_semantic_split{SPLIT_TAG}__ood-{ood_name}.csv"
        df_extra.to_csv(out_extra, index=False)
        print(f"Saved secondary OOD metrics ({ood_name}): {out_extra}")

    # Also write explicit primary OOD-suffixed view for dual-OOD tooling.
    out_primary_suffix = OUT_DIR / f"metrics_semantic_split{SPLIT_TAG}__ood-ood_test.csv"
    results_df.to_csv(out_primary_suffix, index=False)
    print(f"Saved primary OOD-suffixed metrics: {out_primary_suffix}")
else:
    print("WRITE_MINIMAL_OUTPUTS=True: skipped semantic OOD-suffixed metrics files.")



Saved primary OOD metrics: /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_semantic_splitC.csv
Saved secondary OOD metrics (ood_test_injection): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_semantic_splitC__ood-ood_test_injection.csv
Saved secondary OOD metrics (ood_test_injection_standard): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_semantic_splitC__ood-ood_test_injection_standard.csv
Saved primary OOD-suffixed metrics: /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_semantic_splitC__ood-ood_test.csv


In [9]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# 7) Bin-level diagnostics (length + lexical complexity) for TEST/OOD
# Evasion definition: harmful prompt classified as benign (false negative).

# Bin construction reuses shared text/quantile helpers from src.common.notebook_utils.

def _build_bin_rows(stage_name: str, prompts_stage: pd.Series, y_stage: np.ndarray, X_stage, ood_name: str):
    rows = []
    score_stage = sem_clf.predict_proba(X_stage)[:, 1]
    pred_stage = (score_stage >= t_star).astype(int)

    stats_stage = text_stats(prompts_stage)
    stats_stage["length_bin"] = safe_qcut(stats_stage["token_count"], q=4, prefix="len")
    stats_stage["complexity_bin"] = safe_qcut(stats_stage["lexical_ttr"], q=4, prefix="complex")

    for bin_family in ["length_bin", "complexity_bin"]:
        for bin_label in sorted([b for b in stats_stage[bin_family].dropna().unique()]):
            mask = (stats_stage[bin_family] == bin_label).to_numpy(dtype=bool)
            y_s = y_stage[mask]
            p_s = pred_stage[mask]
            s_s = score_stage[mask]

            n_total = int(mask.sum())
            n_pos = int((y_s == 1).sum())
            n_neg = int((y_s == 0).sum())

            if n_total == 0 or n_pos == 0 or n_neg == 0:
                continue

            note = (
                f"bin_eval={bin_family}:{bin_label}; stage={stage_name}; eval_track=deployment_threshold; "
                f"n_total={n_total}; n_pos={n_pos}; n_neg={n_neg}; ood_name={ood_name}"
            )

            res = evaluate_predictions(
                split_name=f"{stage_name}_BIN_{bin_family.upper()}_{str(bin_label).upper()}",
                y_true=y_s,
                y_pred=p_s,
                y_score_for_metrics=s_s,
                print_report=False,
                threshold_note=note,
            )

            evasion_rate = float(((y_s == 1) & (p_s == 0)).sum() / max(n_pos, 1))

            df_one = results_to_dataframe("SEM_LOGREG_BGE", [res])
            df_one["eval_track"] = "deployment_threshold"
            df_one["slice_stage"] = stage_name
            df_one["bin_family"] = bin_family
            df_one["bin_label"] = str(bin_label)
            df_one["bin_count"] = n_total
            df_one["bin_positive_count"] = n_pos
            df_one["bin_negative_count"] = n_neg
            df_one["evasion_rate"] = evasion_rate
            df_one["token_count_median"] = float(np.median(stats_stage.loc[mask, "token_count"].astype(float)))
            df_one["token_count_mean"] = float(np.mean(stats_stage.loc[mask, "token_count"].astype(float)))
            df_one["lexical_ttr_median"] = float(np.median(stats_stage.loc[mask, "lexical_ttr"].astype(float)))
            df_one["avg_token_len_median"] = float(np.median(stats_stage.loc[mask, "avg_token_len"].astype(float)))
            df_one["split_tag"] = SPLIT_TAG
            df_one["ood_name"] = ood_name
            rows.append(df_one)

    return rows

# Primary output (unchanged naming)
bin_rows_primary = []
bin_rows_primary.extend(
    _build_bin_rows("TEST", df_test["prompt_text"].reset_index(drop=True), np.asarray(y_test, dtype=int), X_sem_test, ood_name="id")
)
bin_rows_primary.extend(
    _build_bin_rows("OOD", df_ood_map["ood_test"]["prompt_text"].reset_index(drop=True), np.asarray(y_ood_map["ood_test"], dtype=int), X_sem_ood_map["ood_test"], ood_name="ood_test")
)

if bin_rows_primary:
    semantic_bins_df = pd.concat(bin_rows_primary, ignore_index=True)
    if not WRITE_MINIMAL_OUTPUTS:
        bins_path = OUT_DIR / f"metrics_semantic_bins_split{SPLIT_TAG}.csv"
        semantic_bins_df.to_csv(bins_path, index=False)
        print(f"Saved semantic bin metrics (primary): {bins_path}")
        bins_primary_suffix = OUT_DIR / f"metrics_semantic_bins_split{SPLIT_TAG}__ood-ood_test.csv"
        semantic_bins_df.to_csv(bins_primary_suffix, index=False)
        print(f"Saved semantic bin metrics primary-suffixed: {bins_primary_suffix}")
    else:
        print("WRITE_MINIMAL_OUTPUTS=True: skipped semantic bin metrics file outputs.")
    display(semantic_bins_df.head(12))
else:
    print("No semantic bin metrics were produced for primary OOD.")

# Secondary OOD bin outputs (suffixed)
for ood_name in [k for k in X_sem_ood_map.keys() if k != "ood_test"]:
    rows_extra = _build_bin_rows(
        "OOD",
        df_ood_map[ood_name]["prompt_text"].reset_index(drop=True),
        np.asarray(y_ood_map[ood_name], dtype=int),
        X_sem_ood_map[ood_name],
        ood_name=ood_name,
    )
    if rows_extra and (not WRITE_MINIMAL_OUTPUTS):
        df_extra_bins = pd.concat(rows_extra, ignore_index=True)
        bins_extra_path = OUT_DIR / f"metrics_semantic_bins_split{SPLIT_TAG}__ood-{ood_name}.csv"
        df_extra_bins.to_csv(bins_extra_path, index=False)
        print(f"Saved semantic bin metrics ({ood_name}): {bins_extra_path}")


Saved semantic bin metrics (primary): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_semantic_bins_splitC.csv


Saved semantic bin metrics primary-suffixed: /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_semantic_bins_splitC__ood-ood_test.csv


,model,split,acc,macro_f1,auc_pr,roc_auc,tpr_at_1pct_fpr,tpr_at_5pct_fpr,tpr_at_10pct_fpr,semantic_coverage,...,bin_count,bin_positive_count,bin_negative_count,evasion_rate,token_count_median,token_count_mean,lexical_ttr_median,avg_token_len_median,split_tag,ood_name
0,SEM_LOGREG_BGE,TEST_BIN_LENGTH_BIN_LEN_Q1,0.973684,0.973666,0.997368,0.997230,0.947368,0.947368,1.000000,None,...,38,19,19,0.000000,8.5,7.368421,1.000000,5.111111,C,id
1,SEM_LOGREG_BGE,TEST_BIN_LENGTH_BIN_LEN_Q2,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None,...,37,31,6,0.000000,11.0,10.918919,1.000000,5.100000,C,id
2,SEM_LOGREG_BGE,TEST_BIN_LENGTH_BIN_LEN_Q3,0.918919,0.839827,0.990867,0.943750,0.781250,0.781250,0.781250,None,...,37,32,5,0.062500,14.0,13.783784,0.928571,4.666667,C,id
3,SEM_LOGREG_BGE,TEST_BIN_LENGTH_BIN_LEN_Q4,0.729730,0.703526,0.909268,0.818182,0.346154,0.346154,0.461538,None,...,37,26,11,0.269231,17.0,25.027027,0.916667,4.750000,C,id
4,SEM_LOGREG_BGE,TEST_BIN_COMPLEXITY_BIN_COMPLEX_Q1,0.868421,0.835498,0.977512,0.932143,0.750000,0.750000,0.892857,None,...,38,28,10,0.107143,15.0,21.210526,0.888889,4.700000,C,id
5,SEM_LOGREG_BGE,TEST_BIN_COMPLEXITY_BIN_COMPLEX_Q2,0.864865,0.790960,0.971340,0.895238,0.533333,0.533333,0.533333,None,...,37,30,7,0.100000,14.0,14.864865,0.928571,4.823529,C,id
6,SEM_LOGREG_BGE,TEST_BIN_COMPLEXITY_BIN_COMPLEX_Q3,0.972973,0.826291,1.000000,1.000000,1.000000,1.000000,1.000000,None,...,37,36,1,0.027778,11.0,11.594595,1.000000,5.100000,C,id
7,SEM_LOGREG_BGE,TEST_BIN_COMPLEXITY_BIN_COMPLEX_Q4,0.918919,0.912530,0.930141,0.965839,0.357143,0.928571,0.928571,None,...,37,14,23,0.142857,9.0,9.054054,1.000000,5.375000,C,id
8,SEM_LOGREG_BGE,OOD_BIN_LENGTH_BIN_LEN_Q1,0.692708,0.603681,0.512294,0.829922,0.178571,0.321429,0.428571,None,...,192,28,164,0.250000,7.0,7.208333,1.000000,5.125000,C,ood_test
9,SEM_LOGREG_BGE,OOD_BIN_LENGTH_BIN_LEN_Q2,0.666667,0.666087,0.755604,0.783730,0.285714,0.309524,0.428571,None,...,192,84,108,0.285714,10.0,10.031250,1.000000,4.888889,C,ood_test


Saved semantic bin metrics (ood_test_injection): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_semantic_bins_splitC__ood-ood_test_injection.csv

Saved semantic bin metrics (ood_test_injection_standard): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_semantic_bins_splitC__ood-ood_test_injection_standard.csv


In [ ]:
# Cell Purpose: Execute the next deterministic processing step in this notebook workflow.


<!-- NOTEBOOK_OUTPUT_SUMMARY -->
## 6. Output Summary
1. Writes canonical semantic baseline metrics for the active split to `experiments/results/metrics/metrics_semantic_split{tag}.csv`.
2. When full-output mode is enabled, also writes OOD-specific suffixed semantic metrics and semantic difficulty-bin diagnostics.
3. Produces the semantic baseline rows later consumed by the hybrid repeatability comparison notebook.
